## **Import the packages and get the data**

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import re
import io
import matplotlib.pyplot as plt
import seaborn as sns
import math
from scipy.sparse import csr_matrix
import nltk
nltk.download('wordnet', quiet=True)
from nltk.stem import WordNetLemmatizer
#
from contextlib import redirect_stdout

In [ ]:
# Set display option to show full content of columns
pd.options.display.max_colwidth = 50

# Show all the columns
pd.options.display.max_columns = None

# Turn off scientific notation for pandas DataFrames
pd.options.display.float_format = '{:.2f}'.format

In [4]:
# Take the filtered dataset
file_items_filtered = '/Users/lazr/Desktop/Rec Engine/Datasets/meta_Home_and_Kitchen_filtered.csv'
df_items_filtered = pd.read_csv(file_items_filtered).drop_duplicates()

print(f"df_items_filtered: {len(df_items_filtered):,} rows, {df_items_filtered['asin'].nunique():,} unique ASINs")

/var/folders/86/_khp3pb10vg5vtbr6fmndrxc0000gn/T/ipykernel_5517/3688375420.py:3: DtypeWarning: Columns (0: tech2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_items_filtered = pd.read_csv(file_items_filtered).drop_duplicates()


df_items_filtered: 1,285,392 rows, 1,285,392 unique ASINs


In [5]:
# Take the first 10 rows of the full dataset
file_items_full = '/Users/lazr/Desktop/Rec Engine/Datasets/meta_Home_and_Kitchen.json'
df_items_full = pd.read_json(file_items_full, lines=True, nrows=10)

print(f"df_items_full: {len(df_items_full):,} rows")
df_items_full

df_items_full: 10 rows


,category,tech1,description,fit,title,also_buy,tech2,brand,feature,rank,also_view,main_cat,similar_item,date,price,asin,imageURL,imageURLHighRes
0,"[Home & Kitchen, Kitchen & Dining, Dining & En...",,[It was a time honored tradition among the ear...,,You Are Special Today Red Plate [With Red Pen],"[B0001XR2F2, B01LY51HUN, B07CXZ9C5B, 0310258952]",,Waechtersbach USA,[],"[>#39,665 in Kitchen & Dining (See Top 100 in ...","[B0001XR2F2, B00MOFKX1A, B07G3LN13B, B07CYXMFF...",Amazon Home,"class=""a-bordered a-horizontal-stripes a-spa...","October 8, 2006",$37.00,1487795,[],[]
1,"[Home & Kitchen, Home Dcor, Candles & Holders,...",,[VICKS INHALER relieves stuffy noses helps sin...,,Vicks Inhaler Relief for Cold Sinus Nasal Cong...,[],,Vicks,[],"[>#1,763,185 in Home & Kitchen (See Top 100 in...",[B00UPCRZEC],Amazon Home,,,$4.05,2020300,[],[]
2,"[Home & Kitchen, Kitchen & Dining, Dining & En...",,"[16 oz squeeze bottle, 1 lb.]",,Artistic Churchware Communion Cup Filler: RW525,[],,Artistic Churchware,"[Religious Supply Center, RW-525, Communion Cu...","[>#2,127,003 in Home & Kitchen (See Top 100 in...","[B00C9J79TA, B00XB3ZG5M, B0714K8MSR, 078472530...",Amazon Home,,,$12.48,6564224,[],[]
3,"[Home & Kitchen, Bath, Bathroom Accessories]",,[The only soap in the world with a unique comb...,,4 BARS! Mysore Sandal Soap 70grams FAST SHIPPING,[],,Mysore,[],"[>#6,942,841 in Home & Kitchen (See Top 100 in...",[B001G7PZB0],Amazon Home,,,$22.00,9046461,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
4,"[Home & Kitchen, Home Dcor, Home Fragrance, In...",,"[Divya Arogya vati improves health, It has nat...",,AROGYA VATI (40gm) by popeye seller,[],,Patanjali,[],"[>#3,103,399 in Home & Kitchen (See Top 100 in...","[B075M91MX3, 0258798785, 6565652554]",Amazon Home,,,$5.10,234937912,[],[]
5,"[Home & Kitchen, Wall Art, Posters & Prints]",,[Specifications: About Nikola Tesla: Inventor ...,,Nikola Tesla Photo&hellip;Nikola Tesla Quotes ...,[],,Get Motivated Posters,[],"[>#271,892 in Home & Kitchen (See Top 100 in H...","[B071V2QJGY, B01H0JJ4JG, 4545761445, B078XM4PT...",Amazon Home,"class=""a-bordered a-horizontal-stripes a-spa...",,$14.50,250459655,[],[]
6,"[Home & Kitchen, Home Dcor, Candles & Holders,...",,"[Set of 15 Cello Butterflow Blue, Red & Black ...",,"Set of 15 Cello Butterflow Blue, Red &amp; Bla...",[],,Cello,"[Set of 15 Blue, Red & Black Ball Pen]","[>#1,058,741 in Home & Kitchen (See Top 100 in...","[B00JRZL5FS, B01HMXBCDQ, B0795WCYRV, 067659878...",Amazon Home,,,$7.00,326591516,[],[]
7,"[Home & Kitchen, Wall Art, Posters & Prints]",,[Shiver me timbers! Solve I SPY pirate picture...,,Scholastic Pirate's Treasure Fundle,[0439900581],,Scholastic,[Software: Play I SPY games to uncover the hid...,"6,129 in Software (","[0439042445, 0545415837, 0439763096, 059045846...",Software,,</div>,$2.59,439903491,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
8,"[Home & Kitchen, Bedding, Kids' Bedding, Duvet...",,[<ul><li>100% official merchandise</li><li>Rev...,,My Little Pony Equestria Single/US TWin Duvet ...,"[B00I8TCB02, B00K5B0PCC, B00P8BOYNU, B01AB1CFA...",,My Little Pony,"[100% official merchandise, Features Rainbow D...","[>#1,401,081 in Home & Kitchen (See Top 100 in...","[B072F37S7R, B01GJ90UWI]",Amazon Home,,,,456680012,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...
9,"[Home & Kitchen, Wall Art, Posters & Prints]",,[Great Place to Work institute has been conduc...,,The Great Workplace Poster,[0470559721],,Pfeiffer,[],"[>#3,414,701 in Home & Kitchen (See Top 100 in...",[],Amazon Home,,,$10.89,470902884,[],[]


In [6]:
# Show all columns in the full dataset
print(f"Columns ({len(df_items_full.columns)}):")
for col in df_items_full.columns:
    print(f"  - {col}")

Columns (18):
  - category
  - tech1
  - description
  - fit
  - title
  - also_buy
  - tech2
  - brand
  - feature
  - rank
  - also_view
  - main_cat
  - similar_item
  - date
  - price
  - asin
  - imageURL
  - imageURLHighRes
